# B2-022-probabilistic-latent-models — Practice p16 — Solution

**Type:** proof · **Difficulty:** advanced · **Concepts:** gaussian-reparameterization, variational-autoencoder

*45 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Solution

### (a1) ELBO identity

Fix $x$ with $p(x)>0$. For every $z$ with $q(z\mid x)>0$ and $p(x,z)>0$, Bayes' rule gives $p(z\mid x)=p(x,z)/p(x)$, so
$$\log p(x)=\log p(x,z)-\log p(z\mid x)=\log\frac{p(x,z)}{q(z\mid x)}+\log\frac{q(z\mid x)}{p(z\mid x)} .$$
The left side does not depend on $z$, so taking $E_{z\sim q(\cdot\mid x)}$ of both sides (the expectation of a constant is itself) yields
$$\log p(x)=E_{z\sim q}\Big[\log\frac{p(x,z)}{q(z\mid x)}\Big]+E_{z\sim q}\Big[\log\frac{q(z\mid x)}{p(z\mid x)}\Big]=\mathrm{ELBO}(x)+\mathrm{KL}\big(q(z\mid x)\,\|\,p(z\mid x)\big).$$
(Common support makes each log-ratio finite pointwise, but that alone does not make its expectation finite; a heavy-tailed $q$ can give $\mathrm{KL}=+\infty$. The stated assumption $\mathrm{KL}<\infty$ closes this: since $\log p(x)$ is a finite constant, the ELBO $=\log p(x)-\mathrm{KL}$ is finite too, so the identity is a sum of finite terms.)

### (a2) Bound

By Session 2, $\mathrm{KL}\ge0$, so $\mathrm{ELBO}(x)=\log p(x)-\mathrm{KL}(q\|p(\cdot\mid x))\le\log p(x)$. Equality holds iff the KL is $0$, i.e. iff $q(z\mid x)=p(z\mid x)$ (for almost every $z$): the variational posterior equals the true posterior.

### (a3) Reconstruction minus KL

Using $p(x,z)=p(x\mid z)p(z)$:
$$\mathrm{ELBO}(x)=E_q\big[\log p(x\mid z)\big]+E_q\Big[\log\frac{p(z)}{q(z\mid x)}\Big]=E_q\big[\log p(x\mid z)\big]-\mathrm{KL}\big(q(z\mid x)\,\|\,p(z)\big).$$

### (b1) Pathwise derivatives

$z=\mu+e^{\ell/2}\varepsilon$ with $\varepsilon$ a fixed number. $\partial z/\partial\mu=1$ since $\mu$ enters additively. By the chain rule, $\frac{\partial}{\partial\ell}e^{\ell/2}=e^{\ell/2}\cdot\tfrac12$, so $\partial z/\partial\ell=\tfrac12e^{\ell/2}\varepsilon$.

### (b2) Downstream loss

$\partial L/\partial\mu=L'(z)\cdot1=L'(z)$ and $\partial L/\partial\ell=L'(z)\cdot\tfrac12e^{\ell/2}\varepsilon$.
For $L=z^2$, $\mu=0.3$, $\ell=\log4$ (so $e^{\ell/2}=2$), $\varepsilon=-0.5$: $z=0.3+2(-0.5)=-0.7$ and $L'(z)=2z=-1.4$.
Hence $\partial L/\partial\mu=-1.4$ and $\partial L/\partial\ell=-1.4\cdot\tfrac12\cdot2\cdot(-0.5)=0.7$.

### (b3) Why reparameterize

A sampler that draws $z\sim q(z\mid x)$ directly returns a number that is not expressed as a differentiable function of $\mu$ and $\ell$ — the parameters only shape the distribution the number is drawn from — so automatic differentiation sees no path from the reconstruction term $\log p(x\mid z)$ back to the encoder outputs, and the reconstruction gradient to the encoder is lost (only the closed-form KL would still reach it). Writing $z=\mu+e^{\ell/2}\odot\varepsilon$ with $\varepsilon\sim N(0,I)$ drawn independently of every parameter moves all randomness into $\varepsilon$, so for each draw $z$ is a smooth deterministic function of $(\mu,\ell)$ with the derivatives in (b1), and gradients flow through it. The distribution of $z$ is unchanged: by the Session 1 covariance rule, if $\varepsilon\sim N(0,I)$ then $A\varepsilon+b\sim N(b,AA^\top)$, and with $b=\mu$, $A=\operatorname{diag}(e^{\ell/2})$ this is $N(\mu,\operatorname{diag}(e^{\ell}))=q(z\mid x)$.

## Numeric sanity check

In [ ]:
import math
import torch

# (b2) with autograd
mu = torch.tensor(0.3, dtype=torch.float64, requires_grad=True)
ell = torch.tensor(math.log(4.0), dtype=torch.float64, requires_grad=True)
eps = torch.tensor(-0.5, dtype=torch.float64)
z = mu + torch.exp(0.5 * ell) * eps
(z ** 2).backward()

# (a) identity on a 1-D conjugate model: z ~ N(0,1), x|z ~ N(z,1) => p(x) = N(x;0,2), p(z|x) = N(x/2, 1/2)
def log_normal(v, m, var):
    return -0.5 * (math.log(2 * math.pi * var) + (v - m) ** 2 / var)

x_obs, q_mean, q_var = 1.3, 0.2, 0.8
g = torch.Generator().manual_seed(20260927)
zs = q_mean + math.sqrt(q_var) * torch.randn(400_000, generator=g, dtype=torch.float64)
elbo_mc = (log_normal(x_obs, zs, 1.0) + log_normal(zs, 0.0, 1.0) - log_normal(zs, q_mean, q_var)).mean().item()
post_mean, post_var = x_obs / 2, 0.5
kl_post = 0.5 * (math.log(post_var / q_var) + (q_var + (q_mean - post_mean) ** 2) / post_var - 1)
log_px = log_normal(x_obs, 0.0, 2.0)
elbo_exact_q = (
    -0.5 * math.log(2 * math.pi) - 0.5 * ((x_obs - q_mean) ** 2 + q_var)          # E_q log p(x|z)
    - 0.5 * (q_mean ** 2 + q_var - math.log(q_var) - 1)                          # KL(q || p(z))
)
print(z.item(), mu.grad.item(), ell.grad.item(), elbo_mc, elbo_exact_q, log_px - kl_post)

### Answer check

In [ ]:
assert abs(z.item() + 0.7) < 1e-12
assert abs(mu.grad.item() + 1.4) < 1e-12
assert abs(ell.grad.item() - 0.7) < 1e-12
assert abs(elbo_exact_q - (log_px - kl_post)) < 1e-12        # (a1) and (a3) agree exactly
assert abs(elbo_mc - elbo_exact_q) < 0.01
assert elbo_exact_q <= log_px                                  # (a2)